# 下载 TartanGround（排除 rosbag）

使用官方 `tartanair` 工具下载全部环境、全部轨迹、3 种机器人、12 个相机方向，以及除 `rosbag` 外的所有数据模态。

> 官方完整数据集约 16 TB。排除 rosbag 后仍需准备非常大的磁盘空间。

## 1. 连接 Google Drive

运行下一单元后，按 Colab 提示选择 Google 账号并授权。Notebook 将在 `MyDrive/datasets` 下创建数据集目录。若使用共享云端硬盘，可在配置单元中改为 `/content/drive/Shareddrives/<共享盘名称>/datasets`。

In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT_POINT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)

GOOGLE_DRIVE_DATASETS = DRIVE_MOUNT_POINT / "MyDrive" / "datasets"
GOOGLE_DRIVE_DATASETS.mkdir(parents=True, exist_ok=True)
print(f"Google Drive 数据集根目录: {GOOGLE_DRIVE_DATASETS}")

## 2. 安装官方工具

首次使用时运行。安装完成后如果当前内核仍无法导入 `tartanair`，请重启内核，然后继续运行下面的单元。

In [ ]:
%pip install "tartanair==1.4.0"

## 3. 配置

默认保留 ZIP、不解压。若要解压，请把 `UNZIP` 改为 `True`；只有在解压时才能把 `DELETE_ZIP` 改为 `True`。

In [ ]:
from pathlib import Path

DESTINATION = GOOGLE_DRIVE_DATASETS / "TartanGround"
DATA_SOURCE = "huggingface"  # 可改为 "airlab"
NUM_WORKERS = 4
UNZIP = False
DELETE_ZIP = False

ROBOT_VERSIONS = ["omni", "diff", "anymal"]

# 不要改成空列表：官方工具会把空列表解释为全部模态，包括 rosbag。
MODALITIES_WITHOUT_ROSBAG = [
    "image",
    "meta",
    "depth",
    "seg",
    "lidar",
    "imu",
    "sem_pcd",
    "seg_labels",
    "rgb_pcd",
]

CAMERA_NAMES = [
    "lcam_front", "lcam_right", "lcam_left", "lcam_back", "lcam_top", "lcam_bottom",
    "rcam_front", "rcam_right", "rcam_left", "rcam_back", "rcam_top", "rcam_bottom",
]

assert "rosbag" not in MODALITIES_WITHOUT_ROSBAG
assert not DELETE_ZIP or UNZIP, "DELETE_ZIP=True 时必须同时设置 UNZIP=True"
DESTINATION.mkdir(parents=True, exist_ok=True)

print(f"保存目录: {DESTINATION}")
print(f"数据源: {DATA_SOURCE}")
print(f"机器人: {ROBOT_VERSIONS}")
print(f"模态: {MODALITIES_WITHOUT_ROSBAG}")
print("已确认排除 rosbag。")

## 4. 开始下载

运行此单元将正式开始全量下载。默认的 Hugging Face 下载源会显示每个文件的进度条；若改用 `airlab`，官方工具目前只输出文件开始与完成信息。

In [ ]:
import tartanair as ta
from huggingface_hub.utils import enable_progress_bars

if DATA_SOURCE == "huggingface":
    enable_progress_bars()

ta.init(str(DESTINATION))
ta.download_ground(
    env=[],                 # 全部环境
    version=ROBOT_VERSIONS,
    traj=[],                # 全部轨迹
    modality=MODALITIES_WITHOUT_ROSBAG,
    camera_name=CAMERA_NAMES,
    unzip=UNZIP,
    delete_zip=DELETE_ZIP,
    num_workers=NUM_WORKERS,
    data_source=DATA_SOURCE,
)